### Procesamiento de Lenguaje Natural I
# **Desafío 1**



In [130]:
%pip install numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.


### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups

In [131]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

Utilizamos **20newsgroups** por ser un dataset clásico de NLP ya viene incluido y formateado en sklearn

In [132]:
from sklearn.datasets import fetch_20newsgroups
import numpy as np

## Carga de datos

Cargamos los datos (ya separados de forma predeterminada en train y test)

El dataset 20 Newsgroups contiene aproximadamente 18 000 publicaciones de grupos de noticias distribuidas en 20 temas. Está dividido en dos subconjuntos: uno para entrenamiento (train set) y otro para pruebas (test set).

In [133]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

## Vectorización

Instanciamos un vectorizador.

Podemos ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [134]:
tfidfvect = TfidfVectorizer()

En el atributo `data` accedemos al texto

In [135]:
print(newsgroups_train.data[0])

I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.


Con la interfaz habitual de sklearn podemos ajustar el vectorizador (obtener el vocabulario y calcular el vector IDF) y transformar directamente los datos.

Podemos denominar `X_train` como la matriz documento-término.

In [136]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)

Recordemos que las vectorizaciones por conteos son de tipo sparse, por ello sklearn convenientemente devuelve los vectores de documentos como matrices de tipo sparse.

In [137]:
print(type(X_train))
print(f'shape: {X_train.shape}')
print(f'Cantidad de documentos: {X_train.shape[0]}')
print(f'Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}')

<class 'scipy.sparse._csr.csr_matrix'>
shape: (11314, 101631)
Cantidad de documentos: 11314
Tamaño del vocabulario (dimensionalidad de los vectores): 101631


Una vez ajustado el vectorizador, podemos acceder a atributos como el vocabulario aprendido. Es un diccionario que va de términos a índices.

El índice es la posición en el vector de documento.

In [138]:
tfidfvect.vocabulary_['car']

25775

Probamos con una palbra que no está en el documento.

In [139]:
#tfidfvect.vocabulary_['cocoliso']

Es muy útil tener el diccionario opuesto que va de índices a términos

In [140]:
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

En `y_train` guardamos los targets que son enteros

In [141]:
y_train = newsgroups_train.target
y_train[:10]

array([ 7,  4,  4,  1, 14, 16, 13,  3,  2,  4])

Hay 20 clases correspondientes a los 20 grupos de noticias

In [142]:
print(f'clases {np.unique(newsgroups_test.target)}')
newsgroups_test.target_names

clases [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

## Similaridad de documentos

Veamos similaridad de documentos. Tomemos algún documento

In [143]:
idx = 4811
print(newsgroups_train.data[idx])

THE WHITE HOUSE

                  Office of the Press Secretary
                   (Pittsburgh, Pennslyvania)
______________________________________________________________
For Immediate Release                         April 17, 1993     

             
                  RADIO ADDRESS TO THE NATION 
                        BY THE PRESIDENT
             
                Pittsburgh International Airport
                    Pittsburgh, Pennsylvania
             
             
10:06 A.M. EDT
             
             
             THE PRESIDENT:  Good morning.  My voice is coming to
you this morning through the facilities of the oldest radio
station in America, KDKA in Pittsburgh.  I'm visiting the city to
meet personally with citizens here to discuss my plans for jobs,
health care and the economy.  But I wanted first to do my weekly
broadcast with the American people. 
             
             I'm told this station first broadcast in 1920 when
it reported that year's presidential elec

Medimos la similaridad coseno con todos los documentos de train

In [144]:
cossim = cosine_similarity(X_train[idx], X_train)[0]

Podemos ver los valores de similaridad ordenados de mayor a menor

In [145]:
np.sort(cossim)[::-1]

array([1.        , 0.70930477, 0.67474953, ..., 0.        , 0.        ,
       0.        ], shape=(11314,))

Después vemos a qué documentos corresponden

In [146]:
np.argsort(cossim)[::-1]

array([ 4811,  6635,  4253, ...,  1534, 10055,  4750], shape=(11314,))

Obtenemos los 5 documentos más similares:

In [147]:
mostsim = np.argsort(cossim)[::-1][1:6]
print(mostsim)

[6635 4253 3596 4271 3746]


El documento original pertenece a la clase:

In [148]:
newsgroups_train.target_names[y_train[idx]]

'talk.politics.misc'

Revisamos las clases de los 5 más similares:

In [149]:
for i in mostsim:
  print(newsgroups_train.target_names[y_train[i]])

talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc


### Modelo de clasificación Naïve Bayes

Instanciamos el modelo de clasificación Naive Bayes y lo entrenamos con sklearn

In [150]:
clf = MultinomialNB()
clf.fit(X_train, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


Ya tenemos nuestro vectorizador ya ajustado en train, vectorizamos los textos
del conjunto de test.

In [151]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred =  clf.predict(X_test)

In [152]:
f1_score(y_test, y_pred, average='macro')

0.5854345727938506

### Consigna del Desafío 1

**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado**

**1. Vectorizar documentos**
* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.

**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**
* Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.

**4. Transponer la matriz documento-término.**
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.


El F1-score es una métrica adecuada para evaluar el desempeño de modelos de clasificación, especialmente cuando existe desbalance entre clases.

* El promediado macro calcula el promedio del F1-score de cada clase, otorgando el mismo peso a todas las clases.
* El promediado micro calcula las métricas de forma global considerando todas las predicciones; en problemas de clasificación multiclase suele ser equivalente a la accuracy, por lo que no es la mejor métrica cuando el dataset está desbalanceado.

In [153]:
f1_score(y_test, y_pred, average='macro')

0.5854345727938506

---

### Punto 1 - Vectorizar Documentos

In [154]:
np.random.seed(42)
indices_doc = np.random.choice(len(newsgroups_train.data), size=5, replace=False)

for idx in indices_doc:
    clase_doc = newsgroups_train.target_names[y_train[idx]]
    texto_preview = ' '.join(newsgroups_train.data[idx].split())[:500]

    print(f"\n{'='*120}")
    print(f"Documento Elegido | Indice: {idx} | Clase: {clase_doc}")
    print(f"Contenido Resumido: {texto_preview}")
   

    cossim = cosine_similarity(X_train[idx], X_train)[0]
    arr_mas_similares = np.argsort(cossim)[::-1][1:6]  # Saco el documento mismo y traigo solo los 5 siguientes
    
    # Defino tabla resumen
    print(f"\n{'Indice':<10} {'Similitud':>12} {'Misma Clase':>13} {'Clase':<35} {'Contenido Resumen (250 caracteres)'}")
    print(f"{'-'*10} {'-'*12} {'-'*13} {'-'*35} {'-'*40}")

    aciertos = 0
    for  i in arr_mas_similares:
        clase_similar = newsgroups_train.target_names[y_train[i]]
        sim_score = cossim[i]
        misma_clase = clase_similar == clase_doc
        if misma_clase:
            aciertos += 1
        preview = ' '.join(newsgroups_train.data[i].split())[:500]

        print(f"{i:<10} {sim_score:>12.4f} {'Si' if misma_clase else 'No':>13} {clase_similar:<35} {preview[:500]}...")

    print(f"\n  Documentos de la misma clase entre los 5 mas similares: {aciertos}/5")
    print(f"{'='*120}\n")


Documento Elegido | Indice: 7492 | Clase: comp.sys.mac.hardware
Contenido Resumido: Could someone please post any info on these systems. Thanks. BoB -- ---------------------------------------------------------------------- Robert Novitskey | "Pursuing women is similar to banging one's head rrn@po.cwru.edu | against a wall...with less opportunity for reward"

Indice        Similitud   Misma Clase Clase                               Contenido Resumen (250 caracteres)
---------- ------------ ------------- ----------------------------------- ----------------------------------------
10935            0.6665            Si comp.sys.mac.hardware               Hey everybody: I want to buy a mac and I want to get a good price...who doesn't? So, could anyone out there who has found a really good deal on a Centris 650 send me the price. I don't want to know where, unless it is mail order or areound cleveland, Ohio. Also, should I buy now or wait for the Power PC. Thanks. BoB reply via post or e-ma

#### Conclusiones:

#### Documento #1 - 7492 (comp.sys.mac.hardware)
El documento original es muy corto y genérico , lo que hace que la similitud dependa fuertemente de términos compartidos y de la firma.

Se observa que:

- 3 de los 5 documentos pertenecen a la misma clase.
- El documento más similar comparte firma idéntica, esto explica su alta similitud (0.66).

Se prioriza la coincidencia textual exacta por sobre el contenido semantico de los documentos.

#### Documento #2 - 3546 (comp.os.ms-windows.misc)
En este caso no se recupera ningun documento con la misma clase. Sin embargo vemos que:

- Todos los documentos recuperados pertenecen a hardware.
- Comparten vocabulario técnico (ej: DMA, controladores).
- La categoría original es muy general ("miscellaneous")¡


#### Documento #3 - 5582 (misc.forsale)
Se obtienen buenos resultados:

- 3/5 documentos de la misma clase
- Alta similitud en el top 1

Sin embargo analizando contenido vemos que:

- La similitud está fuertemente influenciada por la firma repetida ("Libertarian, atheist, semi-anarchal Techno-Rat.")

#### Documento #4 - 4793 (talk.politics.guns)
Se observan similitudes bajas (~0.23), solo 2/5 documentos son de la misma clase. 
Los documentos de otras categorías probablemente aparecen por coincidencias lexicas generales


#### Documento #5 - 3813 (sports.hockey)
No se recuperan documentos de la misma clase. De hecho las clases de los documentos recuperados no tienen una relacion evidente con respecto a la clase original. 
Esto evidencia las limitantes de este metodo que tiene limitaciones para capturar el significado semantico de las palabras.

#### Conclusiones generales
Como conclusión general, este ejercicio permite ver las limitaciones del metodo basado en representaciones vectoriales del tipo TF-IDF:
- Es muy sensible a la repeticion de palabras que no aportan valor semantico (Ejemplo: firma de los docuemntos)
- Tiene dificultades para capturar relaciones semanticas

Al no captar de forma adecuada relaciones semanticas, este metodo debe aplicarse en aquellos casos que tengamos vocabularios muy especificos para poder lograr buenos resultados.



### Punto 2

In [155]:
# Clasificacion por prototipo (tipo zero-shot)
sim_matrix = cosine_similarity(X_test, X_train)
indices_mas_similares = np.argmax(sim_matrix, axis=1)
y_pred = y_train[indices_mas_similares]

f1_prototipo = f1_score(y_test, y_pred, average='macro')
print(f"F1-score (macro): {f1_prototipo:.4f}")

F1-score (macro): 0.5050


El clasificador tipo zero-shot obtuvo un F1-Score (macro) de 0.5050. 
Teniendo en cuenta que se trata de un metodo que no requiere entrenamiento y dado que en este problema tenemos 20 categorias el resultado puede considerarse razonable como baseline.
Es importante destacar que este metodo podria tener limitaciones de escalabilidad ya que se compara cada documento de test contra todos los documentos de train (1-N). Ademas, al igual que explicamos en el punto dos, el metodo TF-IDF no permite capturar relaciones semanticas entre los documentos, lo que afecta a los resultados de clasificación.

### Punto 3

In [156]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

# Vectorización
vectorizer = TfidfVectorizer()
X_train_vec = vectorizer.fit_transform(newsgroups_train.data)
X_test_vec = vectorizer.transform(newsgroups_test.data)

# Modelo
model = MultinomialNB()
model.fit(X_train_vec, y_train)

# Predicción
y_pred = model.predict(X_test_vec)

# Evaluación
f1 = f1_score(y_test, y_pred, average='macro')
print("F1:", f1)

F1: 0.5854345727938506


In [157]:
configs = [
    {"model": MultinomialNB, "name": "MNB"},
    {"model": ComplementNB, "name": "CNB"}
]
# Valores de alpha a probar
alphas = [0.01, 0.1, 0.5, 1.0, 2.0]

# Vectorizadores a comparar
vectorizers = [
    {
        "vec": TfidfVectorizer(
            min_df=2, # ignoro palabras que aparecen menos de dos veces
            sublinear_tf=True,  # escalo la frecuencia de las palabras logaritmicamente (1 + log(tf))
        ),
        "name": "TF-IDF"
    },
    {
        "vec": CountVectorizer(
            min_df=2 # ignoro palabras que aparecen menos de dos veces
        ),
        "name": "CountVectorizer"
    }
]

# Loop vectorizadores
for vec_cfg in vectorizers:
    print(f"\n{'='*50}")
    print(f"Vectorizer: {vec_cfg['name']}")
    print(f"{'='*50}")
    
    vectorizer = vec_cfg["vec"]
    
    X_train_vec = vectorizer.fit_transform(newsgroups_train.data)
    X_test_vec = vectorizer.transform(newsgroups_test.data)
    
    # Loop modelos + alphas
    for cfg in configs:
        print(f"\n--- {cfg['name']} ---")
        
        for alpha in alphas:
            model = cfg["model"](alpha=alpha)
            model.fit(X_train_vec, y_train)
            y_pred = model.predict(X_test_vec)
            
            f1 = f1_score(y_test, y_pred, average='macro')
            print(f"alpha={alpha} -> F1: {f1:.4f}")



Vectorizer: TF-IDF

--- MNB ---
alpha=0.01 -> F1: 0.6742
alpha=0.1 -> F1: 0.6716
alpha=0.5 -> F1: 0.6318
alpha=1.0 -> F1: 0.6016
alpha=2.0 -> F1: 0.5675

--- CNB ---
alpha=0.01 -> F1: 0.6743
alpha=0.1 -> F1: 0.6900
alpha=0.5 -> F1: 0.6964
alpha=1.0 -> F1: 0.6932
alpha=2.0 -> F1: 0.6872

Vectorizer: CountVectorizer

--- MNB ---
alpha=0.01 -> F1: 0.6181
alpha=0.1 -> F1: 0.6197
alpha=0.5 -> F1: 0.6136
alpha=1.0 -> F1: 0.5890
alpha=2.0 -> F1: 0.5236

--- CNB ---
alpha=0.01 -> F1: 0.6351
alpha=0.1 -> F1: 0.6385
alpha=0.5 -> F1: 0.6392
alpha=1.0 -> F1: 0.6369
alpha=2.0 -> F1: 0.6335


El mejor resultado se obtuvo utilizando el modelo ComplementNB con un valor de alpha = 0.5 y TfidfVectorizer como representación vectorial de los documentos.

La optimización de los hiperparámetros tanto del modelo como del vectorizador permitió una mejora de aproximadamente 11 puntos porcentuales en F1-score (macro) respecto al modelo baseline.

Los aspectos clave que contribuyeron a esta mejora fueron:

- Configuración del vectorizador:
    - Sublinear TF (sublinear_tf=True): permitió reducir el impacto de términos muy frecuentes mediante una transformación logarítmica de la frecuencia, evitando que ciertas palabras dominen la representación.
    - Filtrado con min_df=2: Elimino palabras que aparecen una sola vez y que aportan poco valor al modelo.
- Elección del modelo ComplementNB:
    - Este modelo tiene mejor desempeño ya que resulta adecuado para problemas de clasificación donde las clases presentan desbalanceos.


### Punto 4

In [158]:
# Transponer matriz
matriz_termino_documento = X_train.T

# Se obtiene vocabulario
vocabulario = tfidfvect.vocabulary_
idx2word = {idx: word for word, idx in vocabulario.items()}

# Palabras a evaluar - Elegidas manualmente
palabras_elegidas = ['car', 'gun', 'music', 'religion', 'hockey']

for palabra in palabras_elegidas:
    
    idx_palabra = vocabulario[palabra]
    
    # Calcular similitud
    similaridad_palabra = cosine_similarity(
        matriz_termino_documento[idx_palabra], 
        matriz_termino_documento
    )[0]

    # Se exluye la posicion 1 yaq que es la misma palabra. Se recuperan las siguientes 5 palabras mas similares
    palabras_mas_similares = np.argsort(similaridad_palabra)[::-1][1:6]

    print(f"\nPalabra: '{palabra}'")
    print(f"{'Palabra similar':<20} {'Similitud':>10}")
    
    for i in palabras_mas_similares:
        print(f"{idx2word[i]:<20} {similaridad_palabra[i]:>10.3f}")


Palabra: 'car'
Palabra similar       Similitud
cars                      0.180
criterium                 0.177
civic                     0.175
owner                     0.169
dealer                    0.168

Palabra: 'gun'
Palabra similar       Similitud
guns                      0.358
crime                     0.244
handgun                   0.239
homicides                 0.233
firearms                  0.233

Palabra: 'music'
Palabra similar       Similitud
inconveniences            0.415
colect                    0.415
posses                    0.415
deaf                      0.408
competencies              0.402

Palabra: 'religion'
Palabra similar       Similitud
religious                 0.245
religions                 0.212
categorized               0.204
purpsoe                   0.201
crusades                  0.199

Palabra: 'hockey'
Palabra similar       Similitud
ncaa                      0.274
nhl                       0.265
affiliates                0.248
xenophobes    

Al transponer la matriz documento-término se obtuvo una representación término-documento, donde cada palabra se representa como un vector que indica su presencia en los distintos documentos. Esto nos permite analizar la simiitud entre las palabras en funcion de su co-ocurrencia en documentos similares.


A partir del análisis realizado, se obtienen los siguientes resultados:

- En algunos casos, el modelo logra capturar correctamente relaciones semanticas entre palabras. Por ejemplo:

        “gun” -> “guns”, “handgun”, “firearms”, “crime”, "homicides"
        “religion” -> “religious”, “religions”, “crusades”
        “hockey” -> “nhl”, “ncaa”, “sportschannel”
        "car" -> "cars", "civic"

- En el caso de "car", vemos que tambien se pudieron capturar palabras que estan relacionadas al dominio general (autos).

- En algunos casos hay presencia de ruido a partir de la captura de palabras que no tienen una relación semantica. Por ejemplo:
        
        “music” → “inconveniences”, “colect”, “posses”
        “hockey” → “xenophobes”

Como conclusion general, podemos afirmar que este tipo de representaciones vectoriales permite sin entrenamiento adicional, capturar relaciones entre las palabras en base a su co-ocurrencia. Sin embargo, esto tiene una gran limitación al no capturar informacion semantica profunda de las palabras, lo que podria llevar a asociaciones ruidosas.